# **Organização e Download Banco de Dados SIAGAS - CPRM**

Como Citar: GOMES. G, C. Gabriel Caldeira Gomes. Script para Organização e Downaload de Banco de Dados - SIAGAS - CPRM. 2026. Disponível em: https://github.com/gabrielcgeo/SIAGAS_BD_Download.

In [ ]:

# Instala as bibliotecas espaciais e a barra de progresso
!pip install geopandas shapely tqdm

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import time
import os
from tqdm.notebook import tqdm
from google.colab import drive

# 1. Monta o Google Drive
drive.mount('/content/drive')

# 2. Cria a pasta 'siagas' no seu Google Drive, se ela não existir
pasta_destino = '/content/drive/MyDrive/siagas'
if not os.path.exists(pasta_destino):
    os.makedirs(pasta_destino)
    print(f"Pasta criada em: {pasta_destino}")
else:
    print(f"Pasta de destino confirmada: {pasta_destino}")

# 3. Define a URL base
url_layer = "https://geoportal.sgb.gov.br/server/rest/services/Siagas_WebMap_MIL1/MapServer/0/query"

# 4. Descobre o tamanho total do banco de dados
print("\nConsultando o tamanho do banco de dados no servidor do CPRM...")
params_count = {
    "where": "1=1",
    "returnCountOnly": "true",
    "f": "json"
}

try:
    response_count = requests.get(url_layer, params=params_count)
    total_registros = response_count.json().get('count', 0)
    print(f"Total de registros encontrados: {total_registros}")
except Exception as e:
    print(f"Erro ao consultar o total: {e}")
    total_registros = 0

# 5. Inicia o download com paginação e barra de progresso
if total_registros > 0:
    offset = 0
    record_count = 1000
    todos_os_registros = []
    continuar_baixando = True

    print("\nIniciando o download dos dados...")

    # Inicia a barra de progresso visual
    pbar = tqdm(total=total_registros, desc="Baixando poços")

    while continuar_baixando:
        params = {
            "where": "1=1",
            "outFields": "*",
            "outSR": "4674",          # <--- MUDANÇA: EPSG 4674 = SIRGAS 2000 Geográfico
            "returnGeometry": "true",
            "f": "json",
            "resultOffset": offset,
            "resultRecordCount": record_count
        }

        try:
            response = requests.get(url_layer, params=params)

            if response.status_code == 200:
                dados = response.json()

                if 'features' in dados and len(dados['features']) > 0:
                    features = dados['features']

                    # Extrai os atributos e a geometria
                    for f in features:
                        atributos = f.get('attributes', {})
                        geometria = f.get('geometry', {})

                        if geometria and 'x' in geometria and 'y' in geometria:
                            # Adiciona as coordenadas capturadas
                            atributos['longitude'] = geometria['x']
                            atributos['latitude'] = geometria['y']

                        todos_os_registros.append(atributos)

                    # Atualiza a barra de progresso com a quantidade baixada neste lote
                    pbar.update(len(features))

                    # Verifica se chegamos ao fim
                    if len(features) < record_count:
                        continuar_baixando = False
                    else:
                        offset += record_count
                        time.sleep(1) # Pausa de 1s para respeitar o servidor
                else:
                    continuar_baixando = False
            else:
                print(f"\nErro no servidor. Código {response.status_code}.")
                continuar_baixando = False

        except Exception as e:
            print(f"\nErro durante o download: {e}")
            continuar_baixando = False

    pbar.close() # Fecha a barra de progresso

    # 6. Processa os dados e salva os arquivos no Drive
    if len(todos_os_registros) > 0:
        print("\nEstruturando os dados e gerando os arquivos...")

        df = pd.DataFrame(todos_os_registros)

        caminho_csv = os.path.join(pasta_destino, 'dados_siagas.csv')
        caminho_gpkg = os.path.join(pasta_destino, 'dados_siagas.gpkg')

        # --- SALVA O CSV ---
        # MUDANÇA: 'decimal=,' converte os pontos flutuantes para vírgulas na exportação
        df.to_csv(caminho_csv, index=False, sep=';', decimal=',', encoding='utf-8-sig')
        print(f"✅ CSV salvo com sucesso: {caminho_csv}")

        # --- SALVA O GPKG ---
        print("Gerando geometria espacial para o GeoPackage... (isso pode levar alguns segundos)")
        geometrias = []
        for idx, row in df.iterrows():
            # Verifica se as coordenadas existem e são válidas
            if 'longitude' in row and 'latitude' in row and pd.notna(row['longitude']) and pd.notna(row['latitude']):
                geometrias.append(Point(row['longitude'], row['latitude']))
            else:
                geometrias.append(Point()) # Cria ponto vazio para dados sem coordenada

        # MUDANÇA: Define o sistema de referências da camada espacial como SIRGAS 2000 (EPSG:4674)
        gdf = gpd.GeoDataFrame(df, geometry=geometrias, crs="EPSG:4674")
        gdf.to_file(caminho_gpkg, driver="GPKG")
        print(f"✅ GeoPackage salvo com sucesso: {caminho_gpkg}")

        print("\nProcesso 100% finalizado! Pode conferir seu Google Drive.")
    else:
        print("\nNenhum dado processado.")
else:
    print("Não foi possível iniciar o download pois o servidor retornou 0 registros.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta de destino confirmada: /content/drive/MyDrive/siagas

Consultando o tamanho do banco de dados no servidor do CPRM...
Total de registros encontrados: 391321

Iniciando o download dos dados...


Baixando poços:   0%|          | 0/391321 [00:00<?, ?it/s]


Estruturando os dados e gerando os arquivos...
✅ CSV salvo com sucesso: /content/drive/MyDrive/siagas/dados_siagas.csv
Gerando geometria espacial para o GeoPackage... (isso pode levar alguns segundos)
✅ GeoPackage salvo com sucesso: /content/drive/MyDrive/siagas/dados_siagas.gpkg

Processo 100% finalizado! Pode conferir seu Google Drive.
